# Module 01e: Plotly Express - Interactive Visualization

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand when to choose Plotly Express over Matplotlib/Seaborn
- Create interactive scatter, line, bar, histogram, box, and violin plots
- Use color, size, facets, and animation to encode multiple dimensions
- Customize layouts, themes, and hover information
- Build geographic (choropleth) and sunburst visualizations
- Export interactive plots as HTML files

**Prerequisites:** Matplotlib fundamentals (01a), Seaborn (01b)

**Datasets:** We use Seaborn's built-in datasets (`tips`, `iris`, `penguins`, `titanic`, `diamonds`, `flights`) so there is nothing extra to download.

---

## 1. Why Plotly Express?

Plotly Express (`px`) is the **high-level** API of the Plotly library. Think of it as the interactive equivalent of Seaborn.

| Feature | Matplotlib | Seaborn | Plotly Express |
|---------|-----------|---------|----------------|
| Interactivity | None | None | Hover, zoom, pan, toggle |
| Code required | Most | Less | Least |
| Output format | Static image | Static image | HTML / embedded widget |
| Best for | Publications | Statistical plots | Dashboards, exploration |
| Pandas integration | Good | Excellent | Excellent |

**Rule of thumb:** Use Plotly Express when you want your audience to *explore* the data themselves.

In [ ]:
# Standard imports
import plotly.express as px
import pandas as pd
import numpy as np
import seaborn as sns

# Load all datasets we will use in this notebook
tips = sns.load_dataset('tips')
iris = sns.load_dataset('iris')
penguins = sns.load_dataset('penguins').dropna()   # drop rows with missing values
titanic = sns.load_dataset('titanic')
diamonds = sns.load_dataset('diamonds').sample(3000, random_state=42)  # sample for speed
flights = sns.load_dataset('flights')

print("Libraries imported and datasets loaded!")
print(f"  tips:     {tips.shape}")
print(f"  iris:     {iris.shape}")
print(f"  penguins: {penguins.shape}")
print(f"  titanic:  {titanic.shape}")
print(f"  diamonds: {diamonds.shape} (sampled)")
print(f"  flights:  {flights.shape}")

### Quick look at the datasets

| Dataset | Rows | Description |
|---------|------|-------------|
| `tips` | 244 | Restaurant bills, tips, and customer info |
| `iris` | 150 | Flower measurements for 3 species |
| `penguins` | ~333 | Body measurements for 3 penguin species |
| `titanic` | 891 | Passenger survival on the Titanic |
| `diamonds` | 3 000 (sampled) | Diamond price, cut, clarity, carat |
| `flights` | 144 | Monthly airline passenger counts 1949-1960 |

---

## 2. The Plotly Express Pattern

Every `px` function follows the same pattern:

```python
fig = px.chart_type(
    data_frame,          # Pandas DataFrame
    x='column_name',     # Column for x-axis
    y='column_name',     # Column for y-axis
    color='column_name', # Color by category or value
    ...                  # Extra dimensions: size, symbol, facet_col, etc.
    title='My Title'
)
fig.show()
```

That's it. One function call produces a fully interactive chart.

---

## 3. Scatter Plots

**When to use:** Explore the relationship between two numeric variables.

**Dataset:** `tips` - restaurant tipping behaviour.

In [ ]:
# 3.1 Basic scatter plot
fig = px.scatter(tips, x='total_bill', y='tip',
                 title='Tip vs Total Bill')
fig.show()

**Try it:** Hover over any point to see its exact values. Click-drag to zoom into a region. Double-click to reset.

### 3.2 Adding Dimensions with Color, Size, and Symbol

Plotly Express can encode up to **6 dimensions** in a single scatter plot:

| Parameter | Encodes | Best for |
|-----------|---------|----------|
| `x` | Horizontal position | Numeric |
| `y` | Vertical position | Numeric |
| `color` | Point colour | Category or numeric |
| `size` | Point size | Numeric |
| `symbol` | Point shape | Category (few levels) |
| `hover_name` | Bold label on hover | Identifier column |

In [ ]:
# 3.2 Multi-dimensional scatter
fig = px.scatter(tips,
                 x='total_bill',
                 y='tip',
                 color='day',          # colour by day of week
                 size='size',          # point size = party size
                 symbol='time',        # shape = Lunch / Dinner
                 title='Tips: 5 Dimensions in One Plot',
                 labels={'total_bill': 'Total Bill ($)',
                         'tip': 'Tip ($)',
                         'size': 'Party Size'})
fig.show()

**Try it:** Click a legend item to hide/show that group. Double-click to isolate a single group.

### 3.3 Scatter with Trendline

Add `trendline='ols'` to overlay a linear regression line (requires `statsmodels`).

In [ ]:
# 3.3 Scatter with OLS trendline, one line per time-of-day
fig = px.scatter(tips,
                 x='total_bill', y='tip',
                 color='time',
                 trendline='ols',
                 title='Tip vs Total Bill with Regression Lines',
                 labels={'total_bill': 'Total Bill ($)', 'tip': 'Tip ($)'})
fig.show()

**Hover over the trendline** to see the OLS equation and R-squared value.

---

## 4. Scatter Matrix (Pair Plot)

**When to use:** Quickly scan relationships between all numeric variable pairs.

**Dataset:** `iris` - flower measurements.

In [ ]:
# 4. Scatter matrix (Plotly's version of Seaborn pairplot)
fig = px.scatter_matrix(iris,
                        dimensions=['sepal_length', 'sepal_width',
                                    'petal_length', 'petal_width'],
                        color='species',
                        title='Iris: Scatter Matrix')
fig.update_traces(diagonal_visible=False)   # hide diagonal histograms
fig.update_layout(height=700, width=700)
fig.show()

**Try it:** Select a rectangular region on any sub-plot to highlight those points across *all* sub-plots. This is called **cross-filtering** and is one of Plotly's most powerful features.

---

## 5. Line Plots

**When to use:** Show trends over time or ordered data.

**Dataset:** `flights` - monthly airline passenger counts (1949-1960).

In [ ]:
# 5.1 Basic line plot - total passengers per year
flights_yearly = flights.groupby('year')['passengers'].sum().reset_index()

fig = px.line(flights_yearly,
              x='year', y='passengers',
              markers=True,
              title='Total Airline Passengers per Year (1949-1960)',
              labels={'passengers': 'Passengers', 'year': 'Year'})
fig.show()

In [ ]:
# 5.2 Multiple lines - one per month to show seasonal patterns
fig = px.line(flights,
              x='year', y='passengers',
              color='month',
              title='Passengers by Month Across Years',
              labels={'passengers': 'Passengers', 'year': 'Year'})
fig.update_layout(hovermode='x unified')   # all months in one hover box
fig.show()

### Key Parameters for `px.line()`

| Parameter | Description | Example |
|-----------|-------------|---------|
| `markers` | Show data-point markers | `True` |
| `line_dash` | Dash style by category | `'sex'` |
| `color` | One line per category | `'month'` |
| `line_group` | Group rows into lines | `'country'` |
| `hover_data` | Extra columns in tooltip | `['col1', 'col2']` |

---

## 6. Bar Charts

**When to use:** Compare values across categories.

**Dataset:** `tips` and `titanic`.

In [ ]:
# 6.1 Simple bar chart - average tip by day
fig = px.bar(tips,
             x='day', y='tip',
             color='sex',
             barmode='group',
             title='Average Tip by Day and Gender',
             labels={'tip': 'Tip ($)', 'day': 'Day of Week'})
fig.show()

### Bar Modes

| `barmode` | Description |
|-----------|-------------|
| `'group'` | Bars side-by-side (default) |
| `'stack'` | Bars stacked on top of each other |
| `'relative'` | Stacked, but negative values go below 0 |

In [ ]:
# 6.2 Stacked bar - Titanic survival by class
survival = (titanic.groupby(['class', 'survived'])
            .size().reset_index(name='count'))
survival['survived'] = survival['survived'].map({0: 'Did Not Survive', 1: 'Survived'})

fig = px.bar(survival,
             x='class', y='count',
             color='survived',
             barmode='stack',
             title='Titanic Survival by Passenger Class',
             labels={'count': 'Number of Passengers', 'class': 'Class'},
             color_discrete_map={'Survived': '#2ecc71', 'Did Not Survive': '#e74c3c'})
fig.show()

In [ ]:
# 6.3 Horizontal bar chart - average bill by day sorted
avg_bill = tips.groupby('day')['total_bill'].mean().reset_index()
avg_bill = avg_bill.sort_values('total_bill')

fig = px.bar(avg_bill,
             x='total_bill', y='day',
             orientation='h',
             title='Average Total Bill by Day',
             labels={'total_bill': 'Average Bill ($)', 'day': ''},
             text_auto='.2f')           # show values on bars
fig.show()

---

## 7. Histograms

**When to use:** Understand the distribution of a single numeric variable.

**Dataset:** `diamonds` - price and carat weight.

In [ ]:
# 7.1 Basic histogram
fig = px.histogram(diamonds,
                   x='price',
                   nbins=50,
                   title='Distribution of Diamond Prices',
                   labels={'price': 'Price (USD)'})
fig.show()

In [ ]:
# 7.2 Overlapping histograms by category
fig = px.histogram(diamonds,
                   x='price',
                   color='cut',
                   nbins=40,
                   barmode='overlay',
                   opacity=0.6,
                   title='Diamond Price Distribution by Cut Quality',
                   labels={'price': 'Price (USD)', 'cut': 'Cut'})
fig.show()

In [ ]:
# 7.3 Marginal distributions on a scatter plot
fig = px.scatter(diamonds,
                 x='carat', y='price',
                 color='cut',
                 marginal_x='histogram',   # histogram on top
                 marginal_y='box',         # box plot on right
                 title='Diamond Price vs Carat with Marginal Distributions',
                 labels={'carat': 'Carat', 'price': 'Price (USD)'})
fig.show()

### Marginal Plot Options

| `marginal_x` / `marginal_y` | Description |
|------------------------------|-------------|
| `'histogram'` | Frequency histogram |
| `'box'` | Box plot |
| `'violin'` | Violin plot |
| `'rug'` | Rug plot (tick marks for every value) |

---

## 8. Box and Violin Plots

**When to use:** Compare distributions across groups and spot outliers.

**Dataset:** `penguins` - body measurements of three penguin species.

In [ ]:
# 8.1 Box plot
fig = px.box(penguins,
             x='species', y='body_mass_g',
             color='sex',
             title='Penguin Body Mass by Species and Sex',
             labels={'body_mass_g': 'Body Mass (g)', 'species': 'Species'})
fig.show()

**Hover** over any box element to see Q1, median, Q3, min, max, and outlier values - much richer than a static box plot.

In [ ]:
# 8.2 Violin plot
fig = px.violin(penguins,
                x='species', y='flipper_length_mm',
                color='sex',
                box=True,              # overlay a mini box plot inside
                points='all',          # show all individual points
                title='Penguin Flipper Length Distribution',
                labels={'flipper_length_mm': 'Flipper Length (mm)',
                        'species': 'Species'})
fig.show()

### `px.violin` Extra Parameters

| Parameter | Description |
|-----------|-------------|
| `box=True` | Show a mini box plot inside the violin |
| `points='all'` | Plot every data point (like swarm) |
| `points='outliers'` | Show only outlier points |
| `points=False` | No individual points |

---

## 9. Heatmaps

**When to use:** Visualise a matrix of values (correlations, pivot tables).

**Dataset:** `flights` - passenger counts per month and year.

In [ ]:
# 9.1 Heatmap from a pivot table
flights_pivot = flights.pivot(index='month', columns='year', values='passengers')

fig = px.imshow(flights_pivot,
                text_auto=True,
                color_continuous_scale='YlOrRd',
                title='Airline Passengers by Month and Year',
                labels={'x': 'Year', 'y': 'Month', 'color': 'Passengers'})
fig.show()

In [ ]:
# 9.2 Correlation heatmap
corr = penguins.select_dtypes(include=[np.number]).corr().round(2)

fig = px.imshow(corr,
                text_auto=True,
                color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1,
                title='Penguin Measurements: Correlation Matrix')
fig.show()

---

## 10. Faceting (Small Multiples)

**When to use:** Create the same chart for each level of a category so patterns jump out.

| Parameter | Description |
|-----------|-------------|
| `facet_col` | Create a column of sub-plots for each category value |
| `facet_row` | Create a row of sub-plots for each category value |
| `facet_col_wrap` | Max columns before wrapping to next row |

**Dataset:** `tips`.

In [ ]:
# 10.1 Faceted scatter - one panel per day
fig = px.scatter(tips,
                 x='total_bill', y='tip',
                 color='smoker',
                 facet_col='day',
                 facet_col_wrap=2,
                 title='Tip vs Total Bill — Faceted by Day',
                 labels={'total_bill': 'Total Bill ($)', 'tip': 'Tip ($)'})
fig.show()

In [ ]:
# 10.2 Faceted histogram - distribution of bill by time (rows) and smoker (columns)
fig = px.histogram(tips,
                   x='total_bill',
                   facet_row='time',
                   facet_col='smoker',
                   color='sex',
                   nbins=20,
                   title='Bill Distribution by Time, Smoker Status, and Gender')
fig.show()

---

## 11. Pie and Sunburst Charts

**When to use:** Show parts of a whole. Sunburst is the hierarchical version of a pie chart.

**Dataset:** `titanic`.

In [ ]:
# 11.1 Pie chart - passenger class breakdown
class_counts = titanic['class'].value_counts().reset_index()
class_counts.columns = ['class', 'count']

fig = px.pie(class_counts,
             names='class', values='count',
             title='Titanic Passengers by Class',
             hole=0.3)    # donut style
fig.show()

In [ ]:
# 11.2 Sunburst - hierarchical breakdown: Class → Sex → Survived
titanic_sb = titanic.copy()
titanic_sb['survived_label'] = titanic_sb['survived'].map({0: 'Died', 1: 'Survived'})

fig = px.sunburst(titanic_sb,
                  path=['class', 'sex', 'survived_label'],
                  title='Titanic: Class → Sex → Survival (click to drill down!)',
                  color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()

**Try it:** Click on any ring segment to zoom into that sub-group. Click the centre to zoom back out.

---

## 12. Customising Plotly Express Figures

Every `px` function returns a `Figure` object. You can customise it further with `.update_layout()`, `.update_traces()`, and `.update_xaxes()` / `.update_yaxes()`.

### 12.1 Themes (Templates)

In [ ]:
# 12.1 Built-in themes
# Available templates: 'plotly', 'plotly_white', 'plotly_dark',
#                      'ggplot2', 'seaborn', 'simple_white', 'presentation'

import plotly.io as pio

templates = ['plotly', 'plotly_white', 'plotly_dark', 'ggplot2', 'seaborn', 'simple_white']

for tmpl in templates:
    fig = px.scatter(iris, x='sepal_length', y='sepal_width', color='species',
                     template=tmpl, title=f"Template: '{tmpl}'")
    fig.update_layout(height=300, width=500)
    fig.show()

### 12.2 Customising Layout and Hover

In [ ]:
# 12.2 Layout customisation
fig = px.scatter(penguins,
                 x='bill_length_mm', y='bill_depth_mm',
                 color='species',
                 hover_data=['body_mass_g', 'island'],
                 title='Penguin Bill Dimensions')

fig.update_layout(
    template='simple_white',
    font_family='Arial',
    title_font_size=20,
    legend_title_text='Penguin Species',
    xaxis_title='Bill Length (mm)',
    yaxis_title='Bill Depth (mm)',
    hoverlabel=dict(bgcolor='white', font_size=12),
)

fig.update_traces(marker=dict(size=8, line=dict(width=1, color='DarkSlateGrey')))

fig.show()

### Common `.update_layout()` Options

| Parameter | Description |
|-----------|-------------|
| `template` | Theme / style |
| `title_font_size` | Title font size |
| `font_family` | Global font |
| `xaxis_title` / `yaxis_title` | Axis labels |
| `legend_title_text` | Legend heading |
| `hovermode` | `'x unified'`, `'closest'`, etc. |
| `height` / `width` | Figure dimensions in pixels |

---

## 13. Color Scales

### Discrete (categorical) palettes

In [ ]:
# 13.1 Discrete colour sequences
sequences = {
    'Plotly (default)': px.colors.qualitative.Plotly,
    'Safe (colorblind)': px.colors.qualitative.Safe,
    'Pastel': px.colors.qualitative.Pastel,
    'Bold': px.colors.qualitative.Bold,
    'Set2': px.colors.qualitative.Set2,
}

for name, seq in sequences.items():
    fig = px.bar(x=list(range(len(seq))), y=[1]*len(seq),
                 color=list(range(len(seq))),
                 color_discrete_sequence=seq,
                 title=f"Discrete palette: {name}")
    fig.update_layout(height=150, showlegend=False,
                      xaxis_visible=False, yaxis_visible=False)
    fig.show()

### Continuous (numeric) colour scales

| Scale | Best for |
|-------|----------|
| `'Viridis'` | General-purpose, colorblind-safe |
| `'RdBu_r'` | Diverging (e.g., correlations from -1 to +1) |
| `'YlOrRd'` | Sequential warm |
| `'Blues'` | Sequential cool |
| `'Plasma'` | High contrast |

**Accessibility tip:** Always prefer `'Safe'` for discrete categories and `'Viridis'` for continuous scales - both are colorblind-friendly.

---

## 14. Animation

**When to use:** Show how data changes over a dimension (often time).

**Dataset:** `iris` - animate through species to compare distributions.

In [ ]:
# 14.1 Animated scatter - cycle through species
fig = px.scatter(iris,
                 x='sepal_length', y='sepal_width',
                 color='species',
                 size='petal_length',
                 animation_frame='species',
                 range_x=[4, 8.5], range_y=[1.5, 4.8],
                 title='Iris: Animated by Species (press Play)')
fig.show()

In [ ]:
# 14.2 Animated bar chart - passenger count by month, animated over years
fig = px.bar(flights,
             x='month', y='passengers',
             color='month',
             animation_frame='year',
             range_y=[0, 650],
             title='Monthly Passengers Animated by Year (press Play)',
             labels={'passengers': 'Passengers', 'month': 'Month'})
fig.update_layout(showlegend=False)
fig.show()

### Animation Tips

| Tip | Why |
|-----|-----|
| Always set fixed `range_x` / `range_y` | Prevents axis jumping between frames |
| Keep frame count reasonable (<30) | Too many frames = slow and confusing |
| Use `animation_group` if rows represent entities | Links same entity across frames |

---

## 15. Saving and Exporting

Plotly figures can be saved as **interactive HTML** or **static images**.

In [ ]:
# 15. Saving figures
fig = px.scatter(iris, x='sepal_length', y='petal_length',
                 color='species', title='Iris (saved example)')

# Save as interactive HTML (anyone can open in a browser)
# fig.write_html('iris_scatter.html')

# Save as static image (requires kaleido: pip install kaleido)
# fig.write_image('iris_scatter.png', scale=2)   # high-res PNG
# fig.write_image('iris_scatter.pdf')             # vector PDF

print("Saving options:")
print("  fig.write_html('file.html')   — Interactive, opens in any browser")
print("  fig.write_image('file.png')   — Static PNG (needs: pip install kaleido)")
print("  fig.write_image('file.pdf')   — Vector PDF (needs: pip install kaleido)")

---

## 16. Plotly Express Cheat Sheet

### Functions at a Glance

| Function | Chart Type | Key Parameters |
|----------|-----------|----------------|
| `px.scatter()` | Scatter plot | `x, y, color, size, symbol, trendline, marginal_x/y` |
| `px.line()` | Line plot | `x, y, color, line_dash, markers` |
| `px.bar()` | Bar chart | `x, y, color, barmode, text_auto, orientation` |
| `px.histogram()` | Histogram | `x, color, nbins, barmode, marginal` |
| `px.box()` | Box plot | `x, y, color, points` |
| `px.violin()` | Violin plot | `x, y, color, box, points` |
| `px.pie()` | Pie / donut | `names, values, hole` |
| `px.sunburst()` | Sunburst | `path, values, color` |
| `px.imshow()` | Heatmap | `text_auto, color_continuous_scale` |
| `px.scatter_matrix()` | Pair plot | `dimensions, color` |
| `px.scatter_3d()` | 3D scatter | `x, y, z, color, size` |

### Universal Parameters (work on most functions)

| Parameter | Description |
|-----------|-------------|
| `color` | Colour by column |
| `facet_col` / `facet_row` | Small multiples |
| `animation_frame` | Animate over a column |
| `hover_data` | Extra columns in tooltip |
| `labels` | Rename axis / legend labels |
| `title` | Chart title |
| `template` | Visual theme |
| `color_discrete_sequence` | Custom palette for categories |
| `color_continuous_scale` | Custom palette for numeric colour |

---

## Summary

In this notebook you learned:

1. **Why Plotly Express?** - Interactive plots with minimal code
2. **Scatter plots** - `px.scatter()` with colour, size, symbol, trendlines, and marginals
3. **Scatter matrix** - `px.scatter_matrix()` for all-pairs exploration
4. **Line plots** - `px.line()` with multiple series and unified hover
5. **Bar charts** - `px.bar()` with grouped, stacked, and horizontal modes
6. **Histograms** - `px.histogram()` with overlays and marginal distributions
7. **Box & violin plots** - `px.box()` and `px.violin()` for distribution comparison
8. **Heatmaps** - `px.imshow()` for matrices and correlations
9. **Faceting** - `facet_col` / `facet_row` for small multiples
10. **Pie & sunburst** - `px.pie()` and `px.sunburst()` for parts-of-whole
11. **Customisation** - Themes, layout, hover, and colour palettes
12. **Animation** - `animation_frame` to animate over a variable
13. **Exporting** - `.write_html()` and `.write_image()`

**Next:** Try the exercises below, then move on to `01d_EDA_Workflow.ipynb` to see how visualisation fits into the full exploratory data analysis process.

---

## Practice Exercises

### Exercise 1: Multi-Dimensional Scatter

Using the **penguins** dataset, create a scatter plot of `flipper_length_mm` vs `body_mass_g`:
- Colour by `species`
- Size by `bill_length_mm`
- Add an OLS trendline
- Use the `'simple_white'` template

In [ ]:
# Your code here


### Exercise 2: Faceted Box Plots

Using the **tips** dataset, create box plots of `total_bill` by `day`:
- Colour by `smoker`
- Facet by `time` (one column per Lunch/Dinner)
- Show all individual points (`points='all'`)

In [ ]:
# Your code here


### Exercise 3: Sunburst Chart

Using the **diamonds** dataset, create a sunburst chart showing the hierarchy: `cut` → `color` → `clarity`.

Hint: use `px.sunburst()` with the `path` parameter.

In [ ]:
# Your code here


### Exercise 4: Animated Histogram

Using the **flights** dataset, create a histogram of `passengers` animated by `year`.
- Use 12 bins
- Fix the y-axis range so it does not jump between frames

In [ ]:
# Your code here


### Exercise 5: Full EDA with Plotly Express

Pick **any** Seaborn dataset (e.g., `penguins`, `titanic`, `diamonds`) and create at least **four** different Plotly Express charts that together tell a story about the data. Include:
1. A distribution plot (histogram or violin)
2. A relationship plot (scatter with trendline)
3. A comparison plot (bar or box)
4. A correlation heatmap

Write a brief markdown summary of your findings after the plots.

In [ ]:
# Your code here
